[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seap-udea/MontuPython/blob/main/examples/MontuPython-HeliacalRises.ipynb)

<p align="left"><img src="https://github.com/seap-udea/MontuPython/raw/main/montu/data/montu-python-logo-complete.webp" width="300" /></p>

# The Heliacal Rise of Sirius, Step by Step

This notebook reconstructs the first morning visibility of Sirius from Thebes around **2782 BCE**, the era of the first *apokatastasis* of the Sothic cycle.

The goal is not to treat `montu.HeliacalRise` as a black box. For each of the four models included in MontuPython we will:

1. obtain the positions of the Sun and Sirius using MontuPython's public positional astronomy routines;
2. write and explain the visibility criterion;
3. evaluate it morning by morning —and, when appropriate, minute by minute—;
4. identify the «not visible → visible» transition that defines the heliacal rise;
5. compare the manual reconstruction with the high-level API.

> **Important interpretation.** A calculated heliacal rise is not a purely geometric instant like a rise or a conjunction. It is a **visibility prediction** and depends on the model, the atmosphere, the horizon, the observer's acuity, and the adopted time step.

If running in Google Colab, MontuPython must be installed first. In a local copy of the repository this cell can remain commented out.

In [1]:
# %pip install -Uq montu

In [2]:
%matplotlib inline
import matplotlib.pyplot as plt
import montu
import numpy as np
import pandas as pd

pd.options.display.float_format = "{:.3f}".format

MontuPython version 0.22.0. 𓇍𓇋𓇋𓏏𓅓𓊵 𓎛𓎡𓄿𓀭𓎛𓈖𓂝𓎡 (ii-ti m Htp, HkAx Hn'-k)


## 1. Setup, conventions, and input data

We need to fix three elements before discussing visibility:

- **Observer:** Thebes (Luxor), from the MontuPython site catalogue.
- **Object:** Sirius, from the bright-star catalogue. Its catalogue visual magnitude is approximately $V=-1.44$.
- **Interval:** July 2782 BCE.

MontuPython uses **astronomical year numbering**: the historical year 2782 BCE is written as `-2781`, because astronomical year 0 exists. We use `calendar='mixed'`, i.e. the Julian calendar before the Gregorian reform.

All altitudes are apparent and expressed in degrees. Azimuth is measured from north toward east ($0°$ north, $90°$ east). JED is the Julian day on the UTC scale that MontuPython uses as a time coordinate.

In [3]:
site = montu.Observer(site="thebes")
sirius = montu.Stars(subset="bright", ProperName="Sirius")
sun = montu.Sun()

start_date = montu.Time("-2781-07-01", calendar="mixed")
end_date = montu.Time("-2781-07-30", calendar="mixed")
V_SIRIUS = float(sirius.data.iloc[0].Vmag)

start_date, end_date

Loading stellar catalogue montu_stellar_catalogue_v38_bright.csv


(Time('-2781-07-01 18:48:41.4'/'-2781-07-01 00:00:00'/'hrw 0-IV-Shemu-17'/JED 705478.5/JTD 705479.2838125),
 Time('-2781-07-30 18:48:41.4'/'-2781-07-30 00:00:00'/'hrw 0-I-Akhet-11'/JED 705507.5/JTD 705508.2838125))

## 2. Models available in MontuPython

MontuPython provides a set of routines to estimate the heliacal rise of a star or planet using four models:

- **Geometric models:**

    - **Ptolemy model (`ptolemy`)**: This is the classical model from Ptolemy's *Almagest*. The critical quantity for morning visibility is the *Arcus Visionis*: the difference between the elevations of the Sun and the observed object above a given horizon (see below) at the exact moment the object rises. This is the most widely used model in archaeoastronomy.

- **Physical models**

    - **Schaefer basic model (`schaefer1987`)**: This model computes heliacal-rise conditions with atmospheric extinction. Essentially, it requires the sky to be dark enough at dawn — defined by the Sun's depression below the horizon and the limiting visual magnitude at the object's altitude.

    - **Schaefer physiological model (`schaefer1985`)**: Similar to the basic model by the same author, but it includes physiological factors of observation for a more realistic estimate. The model has been tested against real observations.

    - **Belokrylov et al. model (`belokrylov2011`)**: This model uses elements of Schaefer's basic approach, but applies corrections derived from real observations for objects of different magnitudes. It is a semi-empirical model and one of the most recent revisions of heliacal-rise theory.

## 3. The `HeliacalRise` interface in MontuPython

The details of all models are implemented in MontuPython in a class, `HeliacalRise`, which computes heliacal rises in a user-friendly way without going deeply into the astronomical and physical details.

### 3.1 Ptolemy's method (Arcus Visionis)

The class is used simply by defining the model and its parameters:

In [4]:
model = montu.HeliacalRise(
    model="ptolemy",
    arcus_visionis_crit=15.0,  # assumed Arcus Visionis
    ptolemy_refraction_deg=34.0/60  # astronomical refraction (if desired)
)

Once the class is defined, we compute the conditions over the chosen date interval:

In [5]:
rises = model.compute(sirius, site, start_date, end_date, verbose=True)
rises

HeliacalRise verbose — model=ptolemy
  quantities: t_rise: local object rise time; AV: solar depression at object rise; AV_crit: critical Arcus Visionis; h_sun: solar altitude
  interval: -2781-07-01 00:00:00 -> -2781-07-30 00:00:00
  model parameters:
    arcus_visionis_crit=15.0
    ptolemy_refraction_deg=0.5666666666666667
  criterion: AV_calc >= AV_crit and h_sun < 0°
  day 001 | -2781-07-01 00:00:00 | visible=False | t_rise=04:54:50.856 | AV=1.561° | AV_crit=15.000° | h_sun=-1.561°
  day 002 | -2781-07-02 00:00:00 | visible=False | t_rise=04:50:54.941 | AV=2.342° | AV_crit=15.000° | h_sun=-2.342°
  day 003 | -2781-07-03 00:00:00 | visible=False | t_rise=04:46:59.026 | AV=3.123° | AV_crit=15.000° | h_sun=-3.123°
  day 004 | -2781-07-04 00:00:00 | visible=False | t_rise=04:43:03.102 | AV=3.904° | AV_crit=15.000° | h_sun=-3.904°
  day 005 | -2781-07-05 00:00:00 | visible=False | t_rise=04:39:07.187 | AV=4.685° | AV_crit=15.000° | h_sun=-4.685°
  day 006 | -2781-07-06 00:00:00 | visib

,model,source,day_jed,jed,local_time,body_altitude_deg,body_azimuth_deg,sun_altitude_deg,sun_azimuth_deg,sun_altitude_formula_deg,vmag,target_horizon_deg,body_ra_hours,body_dec_deg,sun_ra_hours,sun_dec_deg,h_star_deg,h_sun_deg,arcus_visionis_calc_deg,arcus_visionis_crit_deg
0,ptolemy,"Toomer, G. J. (1998). Ptolemy's Almagest. Prin...",705496.500,705496.565,03:44:04.408,-0.567,113.977,-15.586,52.938,-15.586,-1.440,-0.567,3.263,-21.743,6.136,23.992,-79.625,-122.720,15.586,15.000


As you can see, the algorithm returns all detected heliacal rises in the interval as a DataFrame. To inspect the date:

In [6]:
montu.Time(float(rises.jed[0]), format="jd", calendar="mixed")

Time('-2781-07-19 20:22:11.7'/'-2781-07-19 01:33:33'/'hrw 0-I-Mesut-5'/JED 705496.564934/JTD 705497.3487465)

The package includes a method to present the result more clearly:

In [7]:
rises = model.print_rises(rises)

ptolemy — 1 date(s)
  [1] -2781-07-19 00:00:00  03:44:04.408  -2781-06-26 00:00:00.000000  hrw 0-I-Mesut-5  body -0.57°  Sun -15.59°
  source: Toomer, G. J. (1998). Ptolemy's Almagest. Princeton University Press. Book XIII, Chapter 7: "On the heliacal risings and settings of the planets".


Or, if you prefer something a bit more polished:

In [8]:
rises = model.print_rises(rises, title="Heliacal rise of Sirius — Ptolemy", body_label="Sirius")

Heliacal rise of Sirius — Ptolemy — 1 date(s)
  [1] -2781-07-19 00:00:00  03:44:04.408  -2781-06-26 00:00:00.000000  hrw 0-I-Mesut-5  Sirius -0.57°  Sun -15.59°
  source: Toomer, G. J. (1998). Ptolemy's Almagest. Princeton University Press. Book XIII, Chapter 7: "On the heliacal risings and settings of the planets".


Save the result:

In [9]:
rise_ptolemy_jed = rises.day_jed[0]

### 3.2 Schaefer (1987)

The same approach as for the other models, but with a different set of input parameters.

In [10]:
model = montu.HeliacalRise(
    model="schaefer1987",
    k=0.25,  # atmospheric extinction
    limiting_mag_zenith=6.0,  # limiting visual magnitude at the zenith
    sun_depression=-11.0,  # solar depression required for observation
)
rises = model.compute(sirius, site, start_date, end_date, verbose=True)
rises = model.print_rises(rises, title="Heliacal rise of Sirius — Schaefer 1987", body_label="Sirius")

HeliacalRise verbose — model=schaefer1987
  quantities: h_star: object altitude; X: air mass; V_observed: extinguished object magnitude; V_limit: limiting magnitude at object altitude
  interval: -2781-07-01 00:00:00 -> -2781-07-30 00:00:00
  model parameters:
    k=0.25
    limiting_mag_zenith=6.0
    sun_depression=-11.0
  criterion: h_star > 0° and V_observed <= V_limit(local)
  day 001 | -2781-07-01 00:00:00 | visible=False | h_star=-10.681° | X=inf | V_observed=inf | V_limit=-inf
  day 002 | -2781-07-02 00:00:00 | visible=False | h_star=-9.866° | X=inf | V_observed=inf | V_limit=-inf
  day 003 | -2781-07-03 00:00:00 | visible=False | h_star=-9.046° | X=inf | V_observed=inf | V_limit=-inf
  day 004 | -2781-07-04 00:00:00 | visible=False | h_star=-8.222° | X=inf | V_observed=inf | V_limit=-inf
  day 005 | -2781-07-05 00:00:00 | visible=False | h_star=-7.394° | X=inf | V_observed=inf | V_limit=-inf
  day 006 | -2781-07-06 00:00:00 | visible=False | h_star=-6.563° | X=inf | V_observed

Save the result:

In [11]:
rise_schaefer1987_jed = rises.day_jed[0]

### 3.3 Schaefer (1985)

This model scans the twilight. `step_minutes=2` means that visibility is checked every two minutes between astronomical dawn and sunrise.

In [12]:
model = montu.HeliacalRise(
    model="schaefer1985",
    k=0.25,  # atmospheric extinction coefficient
    limiting_mag_zenith=6.0,  # limiting visual magnitude at the zenith
    step_minutes=2.0,  # time step
    twilight_sunbelow=-18.0,  # solar depression
)
rises = model.compute(sirius, site, start_date, end_date, verbose=True)
rises = model.print_rises(rises, title="Heliacal rise of Sirius — Schaefer 1985", body_label="Sirius")

HeliacalRise verbose — model=schaefer1985
  quantities: h_star: object altitude; h_sun: solar altitude; V: object magnitude; V_limit: limiting magnitude at object altitude; B: sky brightness
  interval: -2781-07-01 00:00:00 -> -2781-07-30 00:00:00
  model parameters:
    k=0.25
    limiting_mag_zenith=6.0
    step_minutes=2.0
    twilight_sunbelow=-18.0
  criterion: h_star > 0° and V <= V_limit
  morning scan: h_sun=-18.0° to sunrise, every 2.0 minutes
  day 001 | -2781-07-01 00:00:00 | visible=False
  day 002 | -2781-07-02 00:00:00 | visible=False
  day 003 | -2781-07-03 00:00:00 | visible=False
  day 004 | -2781-07-04 00:00:00 | visible=False
  day 005 | -2781-07-05 00:00:00 | visible=False
  day 006 | -2781-07-06 00:00:00 | visible=False
  day 007 | -2781-07-07 00:00:00 | visible=False
  day 008 | -2781-07-08 00:00:00 | visible=False
  day 009 | -2781-07-09 00:00:00 | visible=False
  day 010 | -2781-07-10 00:00:00 | visible=False
  day 011 | -2781-07-11 00:00:00 | visible=True | h_s

Save the result:

In [13]:
rise_schaefer1985_jed = rises.day_jed[0]

### 3.4 Belokrylov et al. (2011)

This model also scans the twilight. It uses `reference_extinction` as the reference extinction coefficient to correct the star's magnitude.

In [14]:
model = montu.HeliacalRise(
    model="belokrylov2011",
    k=0.25,  # atmospheric extinction coefficient
    reference_extinction=0.25,  # reference extinction coefficient
    step_minutes=2.0,  # time step
)

rises = model.compute(sirius, site, start_date, end_date, verbose=True)
rises = model.print_rises(rises, title="Heliacal rise of Sirius — Belokrylov 2011", body_label="Sirius")

HeliacalRise verbose — model=belokrylov2011
  quantities: h_star: object altitude; h_sun: solar altitude; m': extinction-corrected magnitude; rho: Sun-object separation; h_theor: limiting solar altitude
  interval: -2781-07-01 00:00:00 -> -2781-07-30 00:00:00
  model parameters:
    k=0.25
    reference_extinction=0.25
    step_minutes=2.0
    twilight_sunbelow=-18.0
  criterion: h_star > 0° and h_sun <= h_theor
  morning scan: h_sun=-18.0° to sunrise, every 2.0 minutes
  day 001 | -2781-07-01 00:00:00 | visible=False
  day 002 | -2781-07-02 00:00:00 | visible=False
  day 003 | -2781-07-03 00:00:00 | visible=False
  day 004 | -2781-07-04 00:00:00 | visible=False
  day 005 | -2781-07-05 00:00:00 | visible=False
  day 006 | -2781-07-06 00:00:00 | visible=False
  day 007 | -2781-07-07 00:00:00 | visible=False
  day 008 | -2781-07-08 00:00:00 | visible=False
  day 009 | -2781-07-09 00:00:00 | visible=False
  day 010 | -2781-07-10 00:00:00 | visible=False
  day 011 | -2781-07-11 00:00:00 | 

Save the result:

In [15]:
rise_belokrylov2011_jed = rises.day_jed[0]

## 4. Opening the black box

Next we explain, in basic terms, what the `HeliacalRise.compute` method does internally for each model, so that it is clear how MontuPython's own tools are used for this important calculation, and also which formulas each model applies.

To do this we use two dates in each case: one on which the heliacal rise does not occur and one on which it does, so we can see when the criterion fails and when it is satisfied.

## 4.1. Ptolemy model: *Arcus Visionis*

The source used by MontuPython for this model is: Toomer, G. J. (1998). *Ptolemy's Almagest*. Princeton University Press. Book XIII, Chapter 7: "On the heliacal risings and settings of the planets"

This model asks where the Sun is **when Sirius rises on the eastern horizon** ($h_\star=0$). If $h_\odot$ is the solar altitude at Sirius's rising time, the calculated arc of vision is

$$AV_{\rm calc}\equiv h_\star-h_\odot=-h_\odot$$

Ptolemy's method assumes that Sirius will be visible if $AV_{\rm calc}\ge AV_{\rm crit}$.

MontuPython adopts $AV_{\rm crit}=15°$ for first-magnitude stars. This is a historical threshold, not a consequence of geometry. For other bodies, the value of $AV_{\rm crit}$ changes according to Ptolemy's accumulated experience and his sources:

*   **Venus:** $5^\circ$
*   **Jupiter:** $10^\circ$
*   **Mercury:** $10^\circ$
*   **Saturn:** $11^\circ$
*   **Mars:** $11.5^\circ$

To calculate the Arcus Visionis we need, each day: 1) Sirius's rising time, 2) the Sun's altitude above the horizon, and 3) the Arcus Visionis between the Sun and Sirius at rising. In MontuPython we do it like this:

Let's try first with the day before the heliacal rise. The rising time is:

In [16]:
t_test = montu.Time(rise_ptolemy_jed - 1, format="jd", calendar="mixed")
conditions_sirius = sirius.conditions_in_sky(t_test, site)
jed_rise_star_utc = conditions_sirius.rise_time[0]
site.get_local_time(jed_rise_star_utc)

'03:48:02.509'

Sirius's and the Sun's altitudes at rising are:

In [17]:
mt_rise = montu.Time(jed_rise_star_utc, format="jd", calendar="mixed")
h_sirius = sirius.where_in_sky(mt_rise, site).el[0]
sun.where_in_sky(mt_rise, site)
h_sun = sun.position.el

arcus_visionis_calc = h_sirius - h_sun

h_sirius, h_sun, arcus_visionis_calc

(np.float64(-0.5591616504972591),
 -14.806057195930515,
 np.float64(14.246895545433256))

As we can see, the calculated Arcus Visionis for that day is smaller than the threshold $AV_{\rm crit}=15°$. Therefore that day is not considered the heliacal-rise day.

Now let's try the heliacal-rise day:

In [18]:
t_test = montu.Time(rise_ptolemy_jed, format="jd", calendar="mixed")
conditions_sirius = sirius.conditions_in_sky(t_test, site)
jed_rise_star_utc = conditions_sirius.rise_time[0]

mt_rise = montu.Time(jed_rise_star_utc, format="jd", calendar="mixed")
h_sirius = sirius.where_in_sky(mt_rise, site).el[0]
sun.where_in_sky(mt_rise, site)
h_sun = sun.position.el

arcus_visionis_calc = h_sirius - h_sun

site.get_local_time(jed_rise_star_utc), h_sirius, h_sun, arcus_visionis_calc


('03:44:06.612',
 np.float64(-0.5590967631846221),
 -15.579839081717505,
 np.float64(15.020742318532884))

On this day the Arcus Visionis is already larger, and therefore this is the heliacal rise.

The following day gives:

In [19]:
t_test = montu.Time(rise_ptolemy_jed + 1, format="jd", calendar="mixed")
conditions_sirius = sirius.conditions_in_sky(t_test, site)
jed_rise_star_utc = conditions_sirius.rise_time[0]

mt_rise = montu.Time(jed_rise_star_utc, format="jd", calendar="mixed")
h_sirius = sirius.where_in_sky(mt_rise, site).el[0]
sun.where_in_sky(mt_rise, site)
h_sun = sun.position.el

arcus_visionis_calc = h_sirius - h_sun

site.get_local_time(jed_rise_star_utc), h_sirius, h_sun, arcus_visionis_calc

('03:40:10.722',
 np.float64(-0.5590065446465121),
 -16.352343722129703,
 np.float64(15.79333717748319))

The Arcus Visionis is already greater than the critical value.

It is worth noting that Sirius's altitude at rising is negative because MontuPython's rise and set models include atmospheric refraction, which in a standard atmosphere is close to 34 arcminutes. MontuPython actually computes this refraction from the environmental conditions of the observing site (see the `Observer` class).

## 4.2. Schaefer (1987): fixed solar depression

The source used by MontuPython is Schaefer, B. E. (1987), "Heliacal rise phenomena", *Journal for the History of Astronomy* **18**(11), 19–33.

This model does not wait for Sirius's rising time. It evaluates the sky **once** each morning: when the Sun reaches the chosen depression $h_\odot=-11°$. MontuPython obtains that instant with `Sun.when_is_twilight()`.

For Sirius at altitude $h_\star>0$, the air mass is approximated by the approximation of a plain-parallel atmosphere:

$$X=\csc h_\star=\frac{1}{\sin h_\star}.$$

With catalogue magnitude $V$, extinction coefficient $k$, and zenith limiting magnitude $V_{\rm lim,z}$, the observed magnitude of the star $V_{\rm obs}$ and the limiting magnitude at the height at which it is observed $V_{\rm lim}(h_\star)$ is computed using:

$$V_{\rm obs}=V+kX,$$
$$V_{\rm lim}(h_\star)=V_{\rm lim,z}-k(X-1).$$

Since smaller magnitudes correspond to brighter objects, a margin $V_{\rm lim}-V_{\rm obs}<0$ means that Sirius is still not visible. The criterion for observability is:

$$h_\star>0°\quad\text{and}\quad V_{\rm obs}\le V_{\rm lim}(h_\star).$$

This criterion is fullfilled at many nigths during the year. For finding the heliacal rise, the algorithm sweep several twilights, finding the first where the condition is fulfilled.

Each morning we need: 1) the instant when the Sun is at $-11°$, 2) Sirius's altitude at that instant, and 3) the air mass and the two magnitudes. We start with the day before the heliacal rise:

In [20]:
t_test = montu.Time(rise_schaefer1987_jed - 1, format="jd", calendar="mixed")

# Morning instant with the Sun at -11°
jed_twilight = min(montu.Sun.when_is_twilight(t_test, site, sunbelow=-11.0))
mt_twilight = montu.Time(jed_twilight, format="jd", calendar="mixed")

site.get_local_time(jed_twilight)

'04:08:40.526'

Now we get the elevation of the star:

In [21]:
h_sirius = sirius.where_in_sky(mt_twilight, site).el[0]

k = 0.25
V_lim_zenith = 6.0
X = 1 / np.sin(np.deg2rad(h_sirius))
V_obs = V_SIRIUS + k * X
V_lim = V_lim_zenith - k * (X - 1)

print(f"Tested date: {t_test.readable.datemix}")
print(f"Elevation of Sirius at twilight ({site.get_local_time(jed_twilight)}): {h_sirius:.2f}°")
print(f"Air mass: {X:.2f}")
print(f"Observed magnitude: {V_obs:.2f}")
print(f"Limiting magnitude: {V_lim:.2f}")


Tested date: -2781-07-18 00:00:00
Elevation of Sirius at twilight (04:08:40.526): 3.66°
Air mass: 15.66
Observed magnitude: 2.48
Limiting magnitude: 2.33


The previous day Sirius is already above the horizon when the Sun reaches $-11°$, but $V_{\rm obs}>V_{\rm lim}$: its extinguished magnitude is still too faint compared with the local threshold. That is why the morning does not count as a heliacal rise.

The heliacal-rise day:

In [22]:
t_test = montu.Time(rise_schaefer1987_jed, format="jd", calendar="mixed")

jed_twilight = min(montu.Sun.when_is_twilight(t_test, site, sunbelow=-11.0))
mt_twilight = montu.Time(jed_twilight, format="jd", calendar="mixed")

h_sirius = sirius.where_in_sky(mt_twilight, site).el[0]
sun.where_in_sky(mt_twilight, site)
h_sun = sun.position.el

X = 1 / np.sin(np.deg2rad(h_sirius))
V_obs = V_SIRIUS + k * X
V_lim = V_lim_zenith - k * (X - 1)

print(f"Tested date: {t_test.readable.datemix} ({t_test.readable.datecan})")
print(f"Elevation of Sirius at twilight ({site.get_local_time(jed_twilight)}): {h_sirius:.2f}°")
print(f"Air mass: {X:.2f}")
print(f"Observed magnitude: {V_obs:.2f}")
print(f"Limiting magnitude: {V_lim:.2f}")

Tested date: -2781-07-19 00:00:00 (hrw 0-I-Mesut-5)
Elevation of Sirius at twilight (04:09:01.712): 4.53°
Air mass: 12.67
Observed magnitude: 1.73
Limiting magnitude: 3.08


Now Sirius is higher ($h_\star\approx 4.5°$), the air mass is lower, and the margin becomes positive. That is the first visible morning according to Schaefer (1987).

#4.3. Schaefer (1985): twilight scan

The source used by MontuPython is Schaefer, B. E. (1985), "Predicting Heliacal Risings and Settings", *Sky & Telescope* **70**, $$
dawn ($h_\odot=-18°$) to sunrise in steps of 2 minutes. At each step it recalculates the positions of the Sun and Sirius. If Sirius is below the horizon, it moves to the next step.

The air mass uses Rozenberg's formula, which is more stable near the horizon:

$$X=\frac{1}{\cos z+0.025\exp(-11\cos z)},\qquad z=90°-h_\star.$$

At each step, with $z$, $h_\odot$, and the azimuthal separation $\Delta A$ in radians it compute the following parameter:

$$x=-0.2(V_{\rm lim,z}-7.93+k),$$
$$B_0=79.4(10^x-1)^2-589k,$$
$$L_5=4.75-\frac{\Delta A\,z}{3}+h_\odot(8.2z+12)+2.86z.$$

This parameters allows us to calculate the sky-brightnesss:

$$
B = 
  \begin{cases}
    B_0 + \dfrac{k}{0.20} \cdot 10^{L_5} & \text{si } L_5 \geq -2.07 \\
    B_0 + 589k                          & \text{si } L_5 < -2.07
  \end{cases}
$$

The physiological threshold is

$$E_{\rm th}=C_5(1+\sqrt{K_5B})^2,$$

with $(C_5,K_5)=(1.58\times10^{-10},0.0126)$ if $B<1649$, and $(4.4668\times10^{-9},1.258\times10^{-6})$ otherwise. 

Finally the limiting magnitude is given by,

$$V_{\rm lim}=-16.57-kX-2.5\log_{10}E_{\rm th}.$$

Sirius is visible when $V\le V_{\rm lim}$. The first step that satisfies the criterion is the reported instant.

Let's first define Rozenberg's air mass and the model parameters. Then we evaluate the day before the heliacal rise:

In [23]:
def rozenberg_airmass(h_star):
    z = np.deg2rad(90.0 - h_star)
    cos_z = np.cos(z)
    return 1 / (cos_z + 0.025 * np.exp(-11 * cos_z))

k = 0.25
V_lim_zenith = 6.0
step_minutes = 2.0

t_test = montu.Time(rise_schaefer1985_jed - 1, format="jd", calendar="mixed")
dawn_jed = min(montu.Sun.when_is_twilight(t_test, site, sunbelow=-18.0))
sun.conditions_in_sky(t_test, site)
sunrise_jed = sun.condition.rise_time

print("Scan window:", site.get_local_time(dawn_jed), "→", site.get_local_time(sunrise_jed))

Scan window: 03:28:18.588 → 04:58:42.944


Let's scan:

In [24]:
found = False
for at_jed in np.arange(dawn_jed, sunrise_jed, step_minutes / 1440):
    mt = montu.Time(at_jed, format="jd", calendar="mixed")
    star = sirius.where_in_sky(mt, site).iloc[0]
    if star.el <= 0:
        continue

    # Compute the air mass
    z = np.deg2rad(90.0 - star.el)
    h_sun = np.deg2rad(sun.position.el)
    X = rozenberg_airmass(star.el)

    # Compute the angular separation
    sun.where_in_sky(mt, site)
    delta_az = np.deg2rad(abs((star.az - sun.position.az + 180) % 360 - 180))

    # Compute the parameters for the sky brightness
    x = -0.2 * (V_lim_zenith - 7.93 + k)
    B0 = 79.4 * (10**x - 1)**2 - 589 * k
    L5 = 4.75 - delta_az * z / 3 + h_sun * (8.2 * z + 12) + 2.86 * z
    B = B0 + (k / 0.20) * 10**L5 if L5 >= -2.07 else B0 + 589 * k
    C5, K5 = (1.58e-10, 0.0126) if B < 1649 else (4.4668e-9, 1.258e-6)
    E_th = C5 * (1 + np.sqrt(K5 * B))**2
    V_lim = -16.57 - k * X - 2.5 * np.log10(E_th)
    
    # Check if the star is visible
    visible = V_SIRIUS <= V_lim

    print(
        site.get_local_time(at_jed),
        f"| h_star={star.el:5.2f}°",
        f"| h_sun={sun.position.el:6.2f}°",
        f"| V_lim={V_lim:6.2f}",
        f"| V_star={V_SIRIUS:6.2f}",
        f"| margin={V_lim - V_SIRIUS:6.2f}",
        f"| visible={visible}",
    )
    if visible:
        found = True
        break

print()
print("Was any visible instant found?", found)

04:22:18.588 | h_star= 0.02° | h_sun= -8.04° | V_lim= -3.64 | V_star= -1.44 | margin= -2.20 | visible=False
04:24:18.588 | h_star= 0.43° | h_sun= -7.64° | V_lim= -4.78 | V_star= -1.44 | margin= -3.34 | visible=False
04:26:18.589 | h_star= 0.84° | h_sun= -7.23° | V_lim= -3.70 | V_star= -1.44 | margin= -2.26 | visible=False
04:28:18.590 | h_star= 1.25° | h_sun= -6.82° | V_lim= -2.94 | V_star= -1.44 | margin= -1.50 | visible=False
04:30:18.591 | h_star= 1.66° | h_sun= -6.40° | V_lim= -2.41 | V_star= -1.44 | margin= -0.97 | visible=False
04:32:18.592 | h_star= 2.07° | h_sun= -5.95° | V_lim= -2.05 | V_star= -1.44 | margin= -0.61 | visible=False
04:34:18.593 | h_star= 2.48° | h_sun= -5.48° | V_lim= -1.82 | V_star= -1.44 | margin= -0.38 | visible=False
04:36:18.594 | h_star= 2.89° | h_sun= -4.93° | V_lim= -1.72 | V_star= -1.44 | margin= -0.28 | visible=False
04:38:18.595 | h_star= 3.29° | h_sun= -4.19° | V_lim= -1.76 | V_star= -1.44 | margin= -0.32 | visible=False
04:40:18.588 | h_star= 3.70°

The previous day Sirius does rise during twilight, but in every step $V>V_{\rm lim}$: the sky is still too bright or the star too low. There is no heliacal rise.

Let's repeat the same scan on the heliacal-rise day:

In [25]:
t_test = montu.Time(rise_schaefer1985_jed, format="jd", calendar="mixed")
dawn_jed = min(montu.Sun.when_is_twilight(t_test, site, sunbelow=-18.0))
sun.conditions_in_sky(t_test, site)
sunrise_jed = sun.condition.rise_time

print("Scan window:", site.get_local_time(dawn_jed), "→", site.get_local_time(sunrise_jed))
print()

for at_jed in np.arange(dawn_jed, sunrise_jed, step_minutes / 1440):
    mt = montu.Time(at_jed, format="jd", calendar="mixed")
    star = sirius.where_in_sky(mt, site).iloc[0]
    if star.el <= 0:
        continue

    sun.where_in_sky(mt, site)
    z = np.deg2rad(90.0 - star.el)
    h_sun = np.deg2rad(sun.position.el)
    delta_az = np.deg2rad(abs((star.az - sun.position.az + 180) % 360 - 180))
    X = rozenberg_airmass(star.el)

    x = -0.2 * (V_lim_zenith - 7.93 + k)
    B0 = 79.4 * (10**x - 1)**2 - 589 * k
    L5 = 4.75 - delta_az * z / 3 + h_sun * (8.2 * z + 12) + 2.86 * z
    B = B0 + (k / 0.20) * 10**L5 if L5 >= -2.07 else B0 + 589 * k
    C5, K5 = (1.58e-10, 0.0126) if B < 1649 else (4.4668e-9, 1.258e-6)
    E_th = C5 * (1 + np.sqrt(K5 * B))**2
    V_lim = -16.57 - k * X - 2.5 * np.log10(E_th)
    visible = V_SIRIUS <= V_lim

    print(
        site.get_local_time(at_jed),
        f"| h_star={star.el:5.2f}°",
        f"| h_sun={sun.position.el:6.2f}°",
        f"| V_lim={V_lim:6.2f}",
        f"| V_star={V_SIRIUS:6.2f}",
        f"| margin={V_lim - V_SIRIUS:6.2f}",
        f"| visible={visible}",
    )
    if visible:
        print()
        print("First visible instant → this is the heliacal rise according to Schaefer (1985).")
        break

Scan window: 03:28:25.448 → 04:58:53.519

04:18:25.454 | h_star= 0.03° | h_sun= -8.82° | V_lim= -6.20 | V_star= -1.44 | margin= -4.76 | visible=False
04:20:25.447 | h_star= 0.44° | h_sun= -8.43° | V_lim= -4.61 | V_star= -1.44 | margin= -3.17 | visible=False
04:22:25.448 | h_star= 0.85° | h_sun= -8.05° | V_lim= -3.51 | V_star= -1.44 | margin= -2.07 | visible=False
04:24:25.449 | h_star= 1.26° | h_sun= -7.65° | V_lim= -2.73 | V_star= -1.44 | margin= -1.29 | visible=False
04:26:25.450 | h_star= 1.67° | h_sun= -7.24° | V_lim= -2.18 | V_star= -1.44 | margin= -0.74 | visible=False
04:28:25.451 | h_star= 2.08° | h_sun= -6.83° | V_lim= -1.79 | V_star= -1.44 | margin= -0.35 | visible=False
04:30:25.452 | h_star= 2.49° | h_sun= -6.41° | V_lim= -1.54 | V_star= -1.44 | margin= -0.10 | visible=False
04:32:25.452 | h_star= 2.89° | h_sun= -5.96° | V_lim= -1.39 | V_star= -1.44 | margin=  0.05 | visible=True

First visible instant → this is the heliacal rise according to Schaefer (1985).


The transition is clear: for several minutes the margin is negative, and at the first step with a positive margin the model declares the heliacal rise.

## 4.4. Belokrylov et al. (2011): limiting solar altitude

The source used by MontuPython is Belokrylov, R. O., Belokrylov, S. V., and Nickiforov, M. G. (2011), "Model of the stellar visibility during twilight", *Bulgarian Astronomical Journal* **16**, 50–72, equations (5)–(8).

This model also scans the twilight from $h_\odot=-18°$ to sunrise, but translates visibility into a **limiting solar altitude**. First it corrects the magnitude for extinction (with $k_0=0.25$):

$$m'=V+k(X-1)+(k-k_0).$$

Then it obtains the limiting solar altitude (this comes from empirical fits):

$$h_{\rm lim}=\begin{cases}
-2.47-1.23m', & m'<4.2,\\
15.62-5.61m', & m'\ge4.2.
\end{cases}$$

The Sun–Sirius angular separation is

$$\cos\rho=\sin h_\star\sin h_\odot+\cos h_\star\cos h_\odot\cos(A_\star-A_\odot),$$

and the correction for proximity to the Sun,

$$\Delta h=-0.0338\max(0,58°-\rho),\qquad h_{\rm theor}=h_{\rm lim}+\Delta h.$$

The star is visible if $h_\odot\le h_{\rm theor}$.

Let's evaluate the day before the heliacal rise with the same kind of scan:

In [26]:
k = 0.25
k0 = 0.25
step_minutes = 2.0

t_test = montu.Time(rise_belokrylov2011_jed - 1, format="jd", calendar="mixed")
dawn_jed = min(montu.Sun.when_is_twilight(t_test, site, sunbelow=-18.0))
sun.conditions_in_sky(t_test, site)
sunrise_jed = sun.condition.rise_time

print("Scan window:", site.get_local_time(dawn_jed), "→", site.get_local_time(sunrise_jed))

Scan window: 03:28:18.588 → 04:58:42.944


Let's scan:

In [27]:
found = False
for at_jed in np.arange(dawn_jed, sunrise_jed, step_minutes / 1440):

    mt = montu.Time(at_jed, format="jd", calendar="mixed")
    star = sirius.where_in_sky(mt, site).iloc[0]
    if star.el <= 0:
        continue

    # Compute the air mass
    X = rozenberg_airmass(star.el)
    m_prime = V_SIRIUS + k * (X - 1) + (k - k0)
    h_lim = -2.47 - 1.23 * m_prime if m_prime < 4.2 else 15.62 - 5.61 * m_prime

    # Compute the angular separation
    sun.where_in_sky(mt, site)
    h1, A1 = np.deg2rad(star.el), np.deg2rad(star.az)
    h2, A2 = np.deg2rad(sun.position.el), np.deg2rad(sun.position.az)
    cos_rho = np.sin(h1) * np.sin(h2) + np.cos(h1) * np.cos(h2) * np.cos(A1 - A2)
    rho = np.rad2deg(np.arccos(np.clip(cos_rho, -1, 1)))

    # Compute the correction for proximity to the Sun
    delta_h = -0.0338 * max(0, 58 - rho)
    h_theor = h_lim + delta_h
    visible = sun.position.el <= h_theor

    print(
        site.get_local_time(at_jed),
        f"| h_star={star.el:5.2f}°",
        f"| h_sun={sun.position.el:6.2f}°",
        f"| m'={m_prime:5.2f}",
        f"| h_theor={h_theor:6.2f}°",
        f"| visible={visible}",
    )
    if visible:
        found = True
        break

print()
print("Was any visible instant found?", found)

04:22:18.588 | h_star= 0.02° | h_sun= -8.04° | m'= 8.21 | h_theor=-30.48° | visible=False
04:24:18.588 | h_star= 0.43° | h_sun= -7.64° | m'= 6.49 | h_theor=-20.87° | visible=False
04:26:18.589 | h_star= 0.84° | h_sun= -7.23° | m'= 5.26 | h_theor=-13.95° | visible=False
04:28:18.590 | h_star= 1.25° | h_sun= -6.82° | m'= 4.33 | h_theor= -8.74° | visible=False
04:30:18.591 | h_star= 1.66° | h_sun= -6.40° | m'= 3.61 | h_theor= -6.98° | visible=False
04:32:18.592 | h_star= 2.07° | h_sun= -5.95° | m'= 3.03 | h_theor= -6.27° | visible=False
04:34:18.593 | h_star= 2.48° | h_sun= -5.48° | m'= 2.56 | h_theor= -5.69° | visible=False
04:36:18.594 | h_star= 2.89° | h_sun= -4.93° | m'= 2.17 | h_theor= -5.21° | visible=False
04:38:18.595 | h_star= 3.29° | h_sun= -4.19° | m'= 1.85 | h_theor= -4.81° | visible=False
04:40:18.588 | h_star= 3.70° | h_sun= -3.15° | m'= 1.57 | h_theor= -4.47° | visible=False
04:42:18.588 | h_star= 4.10° | h_sun= -2.66° | m'= 1.33 | h_theor= -4.17° | visible=False
04:44:18.5

The previous day the Sun never becomes low enough relative to $h_{\rm theor}$. There is no heliacal rise.

The heliacal-rise day:

In [28]:
t_test = montu.Time(rise_belokrylov2011_jed, format="jd", calendar="mixed")
dawn_jed = min(montu.Sun.when_is_twilight(t_test, site, sunbelow=-18.0))
sun.conditions_in_sky(t_test, site)
sunrise_jed = sun.condition.rise_time

print("Scan window:", site.get_local_time(dawn_jed), "→", site.get_local_time(sunrise_jed))
print()

for at_jed in np.arange(dawn_jed, sunrise_jed, step_minutes / 1440):
    mt = montu.Time(at_jed, format="jd", calendar="mixed")
    star = sirius.where_in_sky(mt, site).iloc[0]
    if star.el <= 0:
        continue

    sun.where_in_sky(mt, site)
    X = rozenberg_airmass(star.el)
    m_prime = V_SIRIUS + k * (X - 1) + (k - k0)
    h_lim = -2.47 - 1.23 * m_prime if m_prime < 4.2 else 15.62 - 5.61 * m_prime

    h1, A1 = np.deg2rad(star.el), np.deg2rad(star.az)
    h2, A2 = np.deg2rad(sun.position.el), np.deg2rad(sun.position.az)
    cos_rho = np.sin(h1) * np.sin(h2) + np.cos(h1) * np.cos(h2) * np.cos(A1 - A2)
    rho = np.rad2deg(np.arccos(np.clip(cos_rho, -1, 1)))
    delta_h = -0.0338 * max(0, 58 - rho)
    h_theor = h_lim + delta_h
    visible = sun.position.el <= h_theor

    print(
        site.get_local_time(at_jed),
        f"| h_star={star.el:5.2f}°",
        f"| h_sun={sun.position.el:6.2f}°",
        f"| m'={m_prime:5.2f}",
        f"| rho={rho:5.1f}°",
        f"| h_theor={h_theor:6.2f}°",
        f"| visible={visible}",
    )
    if visible:
        print()
        print("First visible instant → this is the heliacal rise according to Belokrylov (2011).")
        break

Scan window: 03:28:25.448 → 04:58:53.519

04:18:25.454 | h_star= 0.03° | h_sun= -8.82° | m'= 8.16 | rho= 56.7° | h_theor=-30.20° | visible=False
04:20:25.447 | h_star= 0.44° | h_sun= -8.43° | m'= 6.46 | rho= 56.7° | h_theor=-20.66° | visible=False
04:22:25.448 | h_star= 0.85° | h_sun= -8.05° | m'= 5.23 | rho= 56.7° | h_theor=-13.79° | visible=False
04:24:25.449 | h_star= 1.26° | h_sun= -7.65° | m'= 4.31 | rho= 56.7° | h_theor= -8.62° | visible=False
04:26:25.450 | h_star= 1.67° | h_sun= -7.24° | m'= 3.59 | rho= 56.7° | h_theor= -6.94° | visible=True

First visible instant → this is the heliacal rise according to Belokrylov (2011).


Here the criterion is read differently: instead of comparing magnitudes, we check whether the Sun's actual altitude is already below the theoretical altitude $h_{\rm theor}$ allowed for that position of Sirius.

## 5. Comparison of the four models

Let's summarize the dates obtained with the `HeliacalRise` interface in section 3. All four models use the same site, the same interval, and the same star; only the visibility criterion changes.

In [29]:
print("Ptolemy")
print("  Date:", montu.Time(rise_ptolemy_jed, format="jd").readable.datemix)
print("  Caniucular:", montu.Time(rise_ptolemy_jed, format="jd").readable.datecan)
print()

print("Schaefer 1987")
print("  Date:", montu.Time(rise_schaefer1987_jed, format="jd").readable.datemix)
print("  Caniucular:", montu.Time(rise_schaefer1987_jed, format="jd").readable.datecan)
print()

print("Schaefer 1985")
print("  Date:", montu.Time(rise_schaefer1985_jed, format="jd").readable.datemix)
print("  Caniucular:", montu.Time(rise_schaefer1985_jed, format="jd").readable.datecan)
print()

print("Belokrylov 2011")
print("  Date:", montu.Time(rise_belokrylov2011_jed, format="jd").readable.datemix)
print("  Caniucular:", montu.Time(rise_belokrylov2011_jed, format="jd").readable.datecan)

Ptolemy
  Date: -2781-07-19 00:00:00
  Caniucular: hrw 0-I-Mesut-5

Schaefer 1987
  Date: -2781-07-19 00:00:00
  Caniucular: hrw 0-I-Mesut-5

Schaefer 1985
  Date: -2781-07-11 00:00:00
  Caniucular: hrw 0-IV-Shemu-27

Belokrylov 2011
  Date: -2781-07-11 00:00:00
  Caniucular: hrw 0-IV-Shemu-27


With the adopted parameters, Ptolemy and Schaefer (1987) place the transition on 19 July 2782 BCE; Schaefer (1985) and Belokrylov et al. (2011) place it on 11 July. The eight-day difference is not a numerical error: each algorithm defines «visible» differently.

In addition, $k$ changes with aerosols and humidity; $V_{\rm lim,z}$ depends on the observer and the night-sky background; an elevated horizon delays the appearance; and clouds or haze can dominate the result. That is why a calculated heliacal-rise date should always be reported together with the model and its parameters.

## 6. Conclusions

We have deliberately separated two layers:

1. **geometry**, computed by MontuPython with `Time`, `Observer`, `Stars.where_in_sky`, `Stars.conditions_in_sky`, `Sun.where_in_sky`, `Sun.when_is_twilight`, and `Sun.conditions_in_sky`;
2. **visibility**, reconstructed in open cells from the equations of each model.

The heliacal rise is the first morning that goes from invisible to visible, not simply the first visible instant found within an interval. The step-by-step reconstructions in section 4 show *why* one day fails the criterion and the next one satisfies it.

This disagreement between models is scientifically informative: it quantifies how «first visibility» depends on the observational model. For a historical study it is advisable always to report site, horizon, atmospheric parameters, model, time step, and calendar system, rather than presenting a calculated date as exact.

### Implemented references

- Schaefer, B. E. (1985), "Predicting Heliacal Risings and Settings", *Sky & Telescope* **70**, 261–263; BASIC listing, lines 34–35 and 55–81.
- Schaefer, B. E. (1987), "Heliacal rise phenomena", *Journal for the History of Astronomy* **18**(11), 19–33.
- Belokrylov, R. O., Belokrylov, S. V., and Nickiforov, M. G. (2011), "Model of the stellar visibility during twilight", *Bulgarian Astronomical Journal* **16**, 50–72, equations (5)–(8).
- Toomer, G. J. (1998), *Ptolemy's Almagest*, Princeton University Press, Book XIII, Chapter 7.

---
*Powered by MontuPython*. For more examples see [MontuPython GitHub repo](https://github.com/seap-udea/MontuPython/tree/main/examples).

Jorge I. Zuluaga © 2023-present